In [2]:
import os, getpass
import asyncio
import nest_asyncio
from typing import List
from dotenv import load_dotenv
import logging

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool as langchain_tool
from langchain.agents import create_tool_calling_agent, AgentExecutor

In [3]:
try:
    # A model with function/tool calling capabilities is required.
    llm = ChatOllama(model="llama3", temperature=0)
    print(f"✅ Language model initialized: {llm.model}")
except Exception as e:
    print(f"❌ Error initializing language model {e}")
    llm = None

✅ Language model initialized: llama3


In [4]:
# --- Define a tool ---
@langchain_tool
def search_information(query: str) -> str:
    """
    Provides factual information on a given topic. Use this tool to find answers to phrses like 'capital of France' or 'weather in London?'
    """
    print(f"\n--- Tool Called: search_information with query: '{query}' ---")
    # simulate a search tool with a dictionary of predefined results.
    simulated_results = {
        "weather in london": "The weather in London is currently cloudy with a temperature of 15ºC.",
        "capital of france": "The capital of France is Paris.",
        "population of earth": "The estimated population of Earth is around 8 billion people.",
        "tallest mountain": "Mount Everest is the tallest mountain above sea level.",
        "default": f"Simulated search result for '{query}': No specific information found, but the topic seems interesting."
    }
    result = simulated_results.get(query.lower(),
                                  simulated_results["default"])
    print(f"--- TOOL RESULT: {result} ---")
    return result    

In [5]:
tools = [search_information]

# --- Create a Tool-Calling Agent ---
if llm:
    # This prompt template requires an `agent_scratchpad` placeholder for the agent's internal steps.
    agent_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant."),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ])

    # Create the agent, binding the LLM, tools, and prompt together.
    agent = create_tool_calling_agent(llm, tools, agent_prompt)

    # AgentExecutor is the runtime that invokesthe agent and executes the chosen tools.
    # the 'tools' argument is not needed here as they are already bound to the agent.
    agent_executor = AgentExecutor(agent=agent, verbose=True,
                                  tools=tools)

async def run_agent_with_tool(query: str):
    """
    Invokes the agent executor with a query and prints the final response."""
    print(f"\n ---🏃🏽 Running Agent with Query: '{query} ---'")
    try:
        response = await agent_executor.ainvoke({"input": query})
        print("\n--- ✅ Final Agent Response ---")
        print(response["output"])
    except Exception as e:
        print(f"\n❌ An error occurred during agent execution: {e}")

In [6]:
async def main():
    """Runs all queries concurently."""
    tasks = [
        run_agent_with_tool("What is the capital of France?"),
        run_agent_with_tool("What's the weather like in London?"),
        run_agent_with_tool("Tell me something about dogs.")
    ]
    await asyncio.gather(*tasks)

nest_asyncio.apply()
asyncio.run(main())


 ---🏃🏽 Running Agent with Query: 'What is the capital of France? ---'

 ---🏃🏽 Running Agent with Query: 'What's the weather like in London? ---'

 ---🏃🏽 Running Agent with Query: 'Tell me something about dogs. ---'


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...

❌ An error occurred during agent execution: registry.ollama.ai/library/llama3:latest does not support tools (status code: 400)

❌ An error occurred during agent execution: registry.ollama.ai/library/llama3:latest does not support tools (status code: 400)

❌ An error occurred during agent execution: registry.ollama.ai/library/llama3:latest does not support tools (status code: 400)
